# STORM: Structural & Tandem-Repeat Optimized Regression Models

This notebook demonstrates end-to-end usage of STORM for association testing of structural variants and tandem repeats.

## 1. Setup and Import

First, ensure the storm package is available. If running from the repo, add the dev path.

In [ ]:
import sys
import subprocess
from pathlib import Path

# Add development path if running from repo
sys.path.insert(0, "../python")

import storm
import polars as pl

print(f"STORM version: {storm.version()}")
print(f"Rust bindings available: {storm.HAS_RUST}")

In [ ]:
# Check available fixtures
fixtures = Path("../fixtures")
print("Available fixtures:")
for f in sorted(fixtures.glob("*")):
    print(f"  {f.name}: {f.stat().st_size} bytes")

## 2. Build Cache from Input Files

STORM builds a cache from:
- Integrated SV VCF (structural variants)
- TRGT VCF (tandem repeat genotypes)
- TRExplorer BED/JSON catalog

In [ ]:
# Build cache from fixtures
cache_dir = "demo_cache"

cache = storm.StormCache.build(
    sv_vcf="../fixtures/sv_small.vcf",
    trgt_vcf="../fixtures/trgt_small.vcf",
    catalog_bed="../fixtures/trexplorer.bed",
    catalog_json="../fixtures/trexplorer.json",
    output_dir=cache_dir,
)

print(f"\nCache built at: {cache_dir}")
print(f"Build stats: {getattr(cache, '_build_stats', 'N/A')}")

## 3. Load and Explore Cache

The cache stores data in Parquet format, which Polars can load efficiently.

In [ ]:
# Load cache (or use existing cache object)
cache = storm.load_cache(cache_dir)

# View test units
print("Test Units:")
print(cache.test_units)

In [ ]:
# View genotypes
print("Genotypes:")
print(cache.genotypes)

In [ ]:
# View catalog
print("Catalog:")
print(cache.catalog)

In [ ]:
# View features
print("Features:")
print(cache.features)

## 4. Explain Genotypes

Get detailed information about resolved genotypes at a locus.

In [ ]:
# Get available test unit IDs
unit_ids = cache.test_units["id"].to_list()
print(f"Available unit IDs: {unit_ids}")

In [ ]:
# Explain a locus (all samples)
unit_id = unit_ids[0]
explanation = storm.explain(cache, unit_id)
print(explanation)

In [ ]:
# Explain for a specific sample
explanation = storm.explain(cache, unit_id, sample_id="SAMPLE1")
print(explanation)

## 5. Run Association Testing

STORM supports multiple models:
- Linear regression for quantitative traits
- Logistic regression for binary traits
- BinomiRare for rare variant burden testing
- Firth regression for rare events

In [ ]:
# Create a dummy phenotype
n_samples = 2
phenotype = pl.Series("phenotype", [0.5, 1.5])

# Run association (placeholder - returns template results)
results = storm.run_glm(
    cache=cache,
    phenotype=phenotype,
)

print("Association Results:")
print(results)

## 6. Cache Verification

Verify the cache structure and integrity.

In [ ]:
# Verify cache
result = storm.verify_cache(cache_dir)
print("Cache verification result:")
for key, value in result.items():
    print(f"  {key}: {value}")

## 7. CLI Usage (Alternative)

STORM also provides a CLI for command-line workflows.

In [ ]:
# Build cache via CLI
!cd .. && cargo run --release -- cache build \
    --sv-vcf fixtures/sv_small.vcf \
    --trgt-vcf fixtures/trgt_small.vcf \
    --catalog-bed fixtures/trexplorer.bed \
    --catalog-json fixtures/trexplorer.json \
    --output-dir /tmp/cli_cache

In [ ]:
# Verify cache via CLI
!cd .. && cargo run --release -- cache verify --cache-dir /tmp/cli_cache

In [ ]:
# Explain via CLI
!cd .. && cargo run --release -- explain sv_sv1 --cache-dir /tmp/cli_cache

## Summary

STORM provides a complete workflow for:

1. **Parsing** - SV VCF, TRGT VCF, TRExplorer catalogs
2. **Resolution** - Mapping SVs to repeat loci, reconstructing diploid lengths
3. **Caching** - Arrow/Parquet storage for efficient access
4. **Analysis** - Multiple regression models and encodings
5. **Output** - Parquet results with full metadata

The Rust core ensures correctness and performance, while the Python API provides interactive exploration via Polars.

In [ ]:
# Cleanup
import shutil
shutil.rmtree(cache_dir, ignore_errors=True)
print("Demo cache cleaned up.")